# AIML – Module 3, Lab 2: Linear Regression, MSE and Polynomial Regression

**Completed practical notebook | Python | scikit-learn**

## Learning objectives
By completing this lab, you will:
1. Understand a linear relationship and fit a simple linear regression model.
2. Compare a brute-force search with a regression model.
3. Calculate Mean Squared Error (MSE) and Root Mean Squared Error (RMSE).
4. Understand why polynomial regression is useful for non-linear data.
5. Observe overfitting when the polynomial degree becomes too high.
6. Apply linear regression to the California Housing dataset and evaluate model performance.

> **How to run:** Open this notebook in Google Colab or Jupyter. Run the cells from top to bottom. The California Housing section downloads its CSV from the public source listed in the notebook, so that section requires an internet connection.

## 1. Import libraries

We use NumPy and pandas for data handling, Matplotlib and Seaborn for visualization, and scikit-learn for regression models and evaluation metrics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import warnings

from sklearn import linear_model, metrics
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Libraries imported successfully.")

## 2. Generate sample linear data

We create 100 input values between 0 and 100. The output is approximately `y = 3x + noise`.

The noise makes the points deviate slightly from a perfect straight line.

In [ ]:
np.random.seed(10)

x = np.linspace(0, 100, 100).reshape(-1, 1)
noise = (np.random.rand(100) * 25).astype(int).reshape(-1, 1)
y = 3 * x + noise

plt.scatter(x, y, color="blue", label="Actual data points")
plt.title("Generated data with an approximately linear relationship")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

## 3. Brute-force approach

A simple approach is to try different values of slope (`m`) and intercept (`c`) and compare the predicted line with the actual data.

For a line:

\[
\hat{y} = mx + c
\]

We calculate the mean absolute error for each candidate line. This is only an illustrative search over a small grid; it is not guaranteed to find the globally best parameters.

In [ ]:
best_m, best_c = None, None
best_error = float("inf")

for m in np.arange(2.5, 3.6, 0.1):
    for c in np.arange(0, 26, 2):
        y_pred_candidate = m * x + c
        error = np.mean(np.abs(y - y_pred_candidate))
        if error < best_error:
            best_error = error
            best_m, best_c = m, c

print(f"Best slope (m): {best_m:.2f}")
print(f"Best intercept (c): {best_c:.2f}")
print(f"Mean absolute error: {best_error:.2f}")

plt.scatter(x, y, color="blue", label="Actual data")
plt.plot(x, best_m * x + best_c, color="red", label="Best grid-search line")
plt.title("Brute-force grid-search result")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

### Limitation of brute-force search

- It may require testing a very large number of combinations.
- The result depends on the search range and step size.
- A coarse search may miss the best parameter values.
- A very fine search is computationally expensive.

Linear regression estimates the best-fitting line more efficiently by minimizing the sum of squared residuals.

## 4. Simple Linear Regression

Linear regression models the relationship between input `x` and output `y` as:

\[
\hat{y} = mx + c
\]

where `m` is the coefficient (slope) and `c` is the intercept.

We split the data into training and testing sets. The model learns from the training data and is evaluated on unseen test data.

In [ ]:
def lin_regression(x_values, y_values, random_state=42, plot=True):
    x_values = np.asarray(x_values).reshape(-1, 1)
    y_values = np.asarray(y_values).reshape(-1, 1)

    x_train, x_test, y_train, y_test = train_test_split(
        x_values, y_values, test_size=0.2, random_state=random_state
    )

    model = linear_model.LinearRegression()
    model.fit(x_train, y_train)

    y_pred = model.predict(x_test)
    mse = metrics.mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = metrics.r2_score(y_test, y_pred)

    print(f"Coefficient (slope): {model.coef_.ravel()[0]:.4f}")
    print(f"Intercept: {model.intercept_.ravel()[0]:.4f}")
    print(f"Test MSE: {mse:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    print(f"Test R² score: {r2:.4f}")

    if plot:
        order = np.argsort(x_test[:, 0])
        plt.scatter(x_values, y_values, color="blue", alpha=0.65, label="Actual data")
        plt.plot(x_test[order], y_pred[order], color="red", linewidth=2, label="Predicted line")
        plt.title("Linear regression: actual data vs predicted line")
        plt.xlabel("x")
        plt.ylabel("y")
        plt.legend()
        plt.show()

    return model, (x_train, x_test, y_train, y_test), y_pred

In [ ]:
linear_model_sample, split_sample, predictions_sample = lin_regression(x, y)

## 5. Mean Squared Error (MSE) and Root Mean Squared Error (RMSE)

Regression models are commonly evaluated by comparing actual values with predicted values.

\[
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
\]

\[
RMSE = \sqrt{MSE}
\]

- **MSE:** Average of the squared prediction errors. Larger errors receive a larger penalty.
- **RMSE:** Square root of MSE. It is expressed in the same unit as the target variable.
- A lower MSE or RMSE generally indicates better prediction accuracy, when comparing models on the same dataset and target scale.

## 6. Why Polynomial Regression?

Linear regression fits a straight line. Some datasets have curved or other non-linear patterns.

Polynomial regression adds powers of the input feature:

\[
\hat{y} = b_0 + b_1x + b_2x^2 + \cdots + b_dx^d
\]

Here, `d` is the polynomial degree. Polynomial regression is still a linear regression model with respect to the coefficients; the input features are transformed into polynomial terms.

In [ ]:
np.random.seed(10)
y2 = 3 * x**2 + np.random.randint(-1000, 1000, size=x.shape)

plt.scatter(x, y2, color="blue", label="Non-linear data with noise")
plt.title("Quadratic data with noise")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

In [ ]:
def poly_regression(x_values, y_values, degree, plot=True):
    x_values = np.asarray(x_values).reshape(-1, 1)
    y_values = np.asarray(y_values).reshape(-1, 1)

    poly = PolynomialFeatures(degree=degree, include_bias=False)
    x_poly = poly.fit_transform(x_values)

    model = linear_model.LinearRegression()
    model.fit(x_poly, y_values)

    y_pred = model.predict(x_poly)
    mse = metrics.mean_squared_error(y_values, y_pred)
    rmse = np.sqrt(mse)
    r2 = metrics.r2_score(y_values, y_pred)

    print(f"Polynomial degree: {degree}")
    print(f"Training MSE: {mse:.4f}")
    print(f"Training RMSE: {rmse:.4f}")
    print(f"Training R² score: {r2:.4f}")

    if plot:
        order = np.argsort(x_values[:, 0])
        plt.scatter(x_values, y_values, color="blue", alpha=0.65, label="Actual data")
        plt.plot(x_values[order], y_pred[order], color="red", linewidth=2,
                 label=f"Polynomial fit (degree {degree})")
        plt.title("Polynomial regression fit")
        plt.xlabel("x")
        plt.ylabel("y")
        plt.legend()
        plt.show()

    return model, poly, y_pred

In [ ]:
# Compare a linear model (degree 1) and a quadratic model (degree 2).
poly_model_degree_1 = poly_regression(x, y2, degree=1)
poly_model_degree_2 = poly_regression(x, y2, degree=2)

### Observation

The quadratic model (degree 2) should represent the quadratic pattern much better than a straight line (degree 1). The reported RMSE here is a **training** error because the model is fitted and evaluated on the same observations. A separate test set is needed to estimate generalization performance.

## 7. Applying Linear Regression to California Housing Data

We now apply regression to a real-world dataset. The dataset contains California district-level information. The target column is `median_house_value`.

**Feature descriptions**
- `longitude`, `latitude`: District location.
- `housing_median_age`: Median house age.
- `total_rooms`, `total_bedrooms`: Room and bedroom counts.
- `population`: District population.
- `households`: Number of households.
- `median_income`: Median income in the district, measured in units of tens of thousands of US dollars.
- `median_house_value`: Median house value in US dollars.
- `ocean_proximity`: Categorical location description.

The original notebook uses `MEDV` as an alias for the target. We retain that alias below for continuity.

In [ ]:
housing_url = (
    "https://raw.githubusercontent.com/ageron/handson-ml/master/"
    "datasets/housing/housing.csv"
)

df = pd.read_csv(housing_url)
df["MEDV"] = df["median_house_value"]

print("Dataset shape:", df.shape)
display(df.head())

### 7.1 Explore the target variable

A histogram shows the distribution of median house values. It helps identify the range, concentration and possible skewness of the target variable.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df["MEDV"], bins=30, kde=True)
plt.title("Distribution of median house values")
plt.xlabel("Median house value (USD)")
plt.ylabel("Number of districts")
plt.show()

### 7.2 Prepare data and inspect correlations

The correlation matrix is calculated only for numeric columns. `ocean_proximity` is categorical, so it is excluded from this numeric correlation analysis.

In [ ]:
numeric_df = df.drop(columns=["ocean_proximity"])
correlation_matrix = numeric_df.corr(numeric_only=True).round(2)

plt.figure(figsize=(11, 8))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation matrix of numeric housing features")
plt.show()

print("Correlation with target (MEDV):")
display(correlation_matrix["MEDV"].sort_values(ascending=False))

### 7.3 Select a feature

The original lab focuses on `median_income`, which typically has a strong positive relationship with median house value. We use it as the single input feature for this simple regression example.

In [ ]:
feature = "median_income"
target = df["MEDV"]

plt.figure(figsize=(9, 5))
plt.scatter(df[feature], target, alpha=0.25)
plt.title("Median income vs median house value")
plt.xlabel("Median income (tens of thousands of USD)")
plt.ylabel("Median house value (USD)")
plt.show()

### 7.4 Train and evaluate the model

We use an 80/20 train-test split and a fixed random state so that the split is reproducible.

- **Training RMSE:** Error on the data used to fit the model.
- **Testing RMSE:** Error on held-out data.
- **R² score:** Measures the proportion of target variance explained by the model (it can be negative on test data if the model performs poorly).

In [ ]:
X = df[[feature]]
y_housing = df["MEDV"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_housing, test_size=0.2, random_state=42
)

housing_model = linear_model.LinearRegression()
housing_model.fit(X_train, y_train)

train_predictions = housing_model.predict(X_train)
test_predictions = housing_model.predict(X_test)

train_rmse = np.sqrt(metrics.mean_squared_error(y_train, train_predictions))
test_rmse = np.sqrt(metrics.mean_squared_error(y_test, test_predictions))
train_r2 = metrics.r2_score(y_train, train_predictions)
test_r2 = metrics.r2_score(y_test, test_predictions)

print("Training set performance")
print("------------------------")
print(f"RMSE: {train_rmse:,.2f} USD")
print(f"R²:   {train_r2:.4f}")

print("\nTesting set performance")
print("-----------------------")
print(f"RMSE: {test_rmse:,.2f} USD")
print(f"R²:   {test_r2:.4f}")

print("\nModel coefficient:", housing_model.coef_[0])
print("Model intercept:", housing_model.intercept_)

In [ ]:
# Visualize the fitted line against the test observations.
order = np.argsort(X_test[feature].to_numpy())

plt.figure(figsize=(9, 5))
plt.scatter(X_test[feature], y_test, alpha=0.25, label="Actual test values")
plt.plot(
    X_test[feature].to_numpy()[order],
    test_predictions[order],
    color="red",
    linewidth=2,
    label="Predicted regression line"
)
plt.title("California housing: test data and linear regression")
plt.xlabel("Median income (tens of thousands of USD)")
plt.ylabel("Median house value (USD)")
plt.legend()
plt.show()

### Interpretation of the housing model

1. The model uses median income as its only predictor.
2. A positive coefficient indicates that higher median income is associated with higher predicted median house value in this one-feature model.
3. RMSE is the typical scale of prediction error, expressed in US dollars.
4. The test metrics provide a better indication of performance on unseen observations than training metrics.
5. This is a simple baseline model. It does not use all available features, and it should not be interpreted as a causal relationship.

# Exercises – Completed Answers

## Exercise 1: Higher Degree Polynomials

**Question:** What happens when a very high-degree polynomial is fitted to noisy, non-linear data?

**Answer:** A high-degree polynomial is flexible and can bend to pass close to many training observations, including random noise and outliers. It may achieve very low training error but perform poorly on new data. This is called **overfitting**.

The cell below uses the noisy sine-wave dataset and lets you change the polynomial degree from 2 to 15. Compare the fitted curve and training RMSE for different degrees.

In [ ]:
# Change degree to experiment with polynomial complexity (2 to 15).
degree_exercise_1 = 4

np.random.seed(10)
x_sine = np.array([i * np.pi / 180 for i in range(60, 300, 6)])
y_sine = np.sin(x_sine) + np.random.normal(0, 0.15, len(x_sine))

exercise_model_1 = poly_regression(x_sine, y_sine, degree=degree_exercise_1)

### Exercise 1 – Conclusion

As polynomial degree increases, the model becomes more flexible. Very high degrees can fit random fluctuations in the training data rather than the underlying sine-wave pattern. This is overfitting: training performance may look good, but generalization to unseen data can become worse.

## Exercise 2: Run the polynomial model with degree 1

**Question:** What do you observe when the polynomial degree is set to 1?

**Answer:** Degree 1 is equivalent to ordinary linear regression. It can only fit a straight line. Since the sine-wave pattern is curved, a straight line cannot represent the full pattern well and generally produces a larger training error than a suitable higher-degree polynomial.

Run the cell below to see the degree-1 fit.

In [ ]:
degree_exercise_2 = 1

np.random.seed(10)
x_sine_linear = np.array([i * np.pi / 180 for i in range(60, 300, 6)])
y_sine_linear = np.sin(x_sine_linear) + np.random.normal(0, 0.15, len(x_sine_linear))

exercise_model_2 = poly_regression(x_sine_linear, y_sine_linear, degree=degree_exercise_2)

### Exercise 2 – Conclusion

A degree-1 polynomial produces a straight line and underfits the curved sine-wave data. A higher degree can capture curvature, but an excessively high degree may overfit noise. The aim is to select a model complexity that generalizes well, not simply the model with the smallest training error.

## Overall Lab Conclusion

- Linear regression models a straight-line relationship between input and output.
- MSE and RMSE measure prediction error.
- Polynomial regression models curved relationships by transforming input features into polynomial terms.
- Very high polynomial degrees can cause overfitting.
- A train-test split helps evaluate how well a model performs on unseen data.
- The California Housing example demonstrates a simple real-world regression workflow using median income to predict median house value.

## References

1. [California Housing dataset source](https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.csv)
2. [scikit-learn Linear Regression documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
3. [scikit-learn PolynomialFeatures documentation](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)
4. [scikit-learn model evaluation documentation](https://scikit-learn.org/stable/modules/model_evaluation.html)